# Week 2, day 4 (afternoon) — Worksheet 08: The structures at work — data engineering and analytics

Everything so far taught the structures. This sheet is about **choosing**
between them, on the kind of work you would actually be paid to do:
deduplicating a feed, validating a schema, joining a fact table to a
dimension, aggregating, and reporting data quality.

`raw_events` is a list of dictionaries — the shape data arrives in from an API
or a JSON log. It is deliberately dirty: it contains an exact duplicate, a
record missing a field, a user ID that is not in the lookup table, and some
null amounts. Every one of those is a thing you will meet in week one of a
real job.

Run the setup cell once, then work down. **Question 2 ends in a deliberate
error.** The questions run in order and later ones use `clean` from Q3.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 08 — The structures at work. Run this once.
from array import array

# One dict per record, exactly as a JSON feed would hand it to you.
# This data is deliberately dirty -- see the intro.
raw_events = [
    {"event_id": "e1", "user": "u01", "action": "view",     "amount": None,    "ts": "2026-08-20"},
    {"event_id": "e2", "user": "u02", "action": "purchase", "amount": "49.99", "ts": "2026-08-20"},
    {"event_id": "e3", "user": "u01", "action": "purchase", "amount": "12.50", "ts": "2026-08-21"},
    {"event_id": "e2", "user": "u02", "action": "purchase", "amount": "49.99", "ts": "2026-08-20"},
    {"event_id": "e4", "user": "u03", "action": "view",     "amount": None,    "ts": "2026-08-21"},
    {"event_id": "e5", "user": "u99", "action": "purchase", "amount": "5.00",  "ts": "2026-08-22"},
    {"event_id": "e6", "user": "u01", "action": "refund",   "amount": "12.50", "ts": "2026-08-22"},
    {"event_id": "e7", "user": "u02", "action": "purchase",                    "ts": "2026-08-23"},
]

# The dimension table you join against. Note who is NOT in it.
users = {
    "u01": {"name": "Ada", "region": "East"},
    "u02": {"name": "Bo",  "region": "West"},
    "u03": {"name": "Cai", "region": "East"},
}

# The contract every record is supposed to satisfy.
REQUIRED_FIELDS = {"event_id", "user", "action", "amount", "ts"}

print("raw records:", len(raw_events))
print("known users:", len(users))

PART A — Picking the right structure

The same column, four ways. Each answers a different question, and picking
wrong is how a report ends up quietly meaning something else.

### Question 1

Pull the `user` field out of every record into a **list**. Print it and its length. Then build a **set** from it and print that with its length. Then build a **dict** counting how many events each user has. Say in a comment which of the three you would use to answer "how many users do we have" and which for "who is our busiest user".
> **HINT:** the list keeps every occurrence, the set keeps each user once, the dict keeps a number per user. Three different questions.

In [ ]:
############################
## Your Code Here
############################

### Question 2

A list holds anything; an **array** holds one type and says so. Build `amounts` as an `array` of doubles (`"d"`) from `[49.99, 12.50, 5.00]` and print it, its length and its `typecode`. Append `1.25` and print it again. Then try to append the string `"free"` and let it fail.
> **CAREFUL:** the last line is meant to raise. That refusal is the whole point of an array — it is what a list would never do for you.

In [ ]:
############################
## Your Code Here
############################

PART B — Ingestion: cleaning the feed

### Question 3

`raw_events` contains one exact duplicate. Deduplicate on `event_id`, **keeping the original order**, into a list called `clean`. Print how many records you started with, how many you kept, and which `event_id` was dropped.
> **HINT:** this is the `seen` set pattern. A `set()` alone would deduplicate but destroy the order, and you cannot put a dict in a set anyway.

In [ ]:
############################
## Your Code Here
############################

### Question 4

Validate every record in `clean` against `REQUIRED_FIELDS`. For each record print its `event_id` and the set of fields it is **missing**; then print a list of just the `event_id`s that failed. A record with no missing fields should report an empty set.
> **HINT:** `set(a_dict)` gives you its keys. Missing fields are then one set difference — no loop over field names needed.

In [ ]:
############################
## Your Code Here
############################

### Question 5

The `amount` field arrives as a string, or is `None`, or is absent entirely. Build `parsed`: a list of `(event_id, action, amount_as_float_or_None)` tuples over `clean`. Print it, then print the total of the amounts that are not `None`.
> **CAREFUL:** one record has no `amount` key at all, so `event["amount"]` would raise. Reach for `.get()`.

In [ ]:
############################
## Your Code Here
############################

PART C — Joining to a dimension

### Question 6

Enrich each record in `clean` with its user's name and region from the `users` lookup, printing one line per record as `e1 Ada East`. Records whose user is not in `users` should print `UNKNOWN UNKNOWN` rather than crash.
> **HINT:** a dictionary IS the lookup — `users[uid]` is the join. Use `.get()` with a fallback dict so an unmatched key does not stop the pipeline.

In [ ]:
############################
## Your Code Here
############################

### Question 7

Find the referential-integrity problem. Print the set of user IDs appearing in `clean`, the set of user IDs in `users`, and the users that appear in the events but are **missing from the lookup**. Then print how many records those orphans account for.
> **HINT:** "in the facts but not the dimension" is a set difference — the anti-join you wrote with `NOT EXISTS` in week 2, day 2.

In [ ]:
############################
## Your Code Here
############################

PART D — Aggregating

### Question 8

Group `parsed` by action. Print a dictionary of how many records each action has, and a second dictionary of the total amount per action, treating `None` as zero. Then print both sorted by value, biggest first.

In [ ]:
############################
## Your Code Here
############################

### Question 9

Count **distinct users** per action — not records, distinct users. Build a dictionary mapping each action to a set of user IDs, print it, then print the count per action. Compare those counts with the record counts from Q8.
> **HINT:** the value is a `set()` and you `add` to it, so a user doing the same action twice cannot inflate the number.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Compute **net revenue per user**: purchases add, refunds subtract, views count as zero. Print the dictionary, then the users ranked by net revenue, biggest first. Look carefully at the user who nets zero.
> **HINT:** one loop over `clean`, an `if` on the action, and `.get()` for the amount — everything you need is from Q5 and Q8.

In [ ]:
############################
## Your Code Here
############################

PART E — Stretch: the data quality report

The cell nobody writes and everybody needs. It is also where the numbers on
this sheet stop agreeing with each other.

### Question 11

Stretch. In one cell print a quality summary: rows in, rows after dedup, duplicates dropped, records failing schema, orphan user IDs, records with a null or missing amount, and **three different revenue figures** — the sum of every amount, the gross purchase value, and net revenue after refunds. Then answer in a comment: which numbers on this sheet would mislead someone, and why.
> **CAREFUL:** you already computed a "total" in Q5. Before you print it here, work out whether it is revenue.

In [ ]:
############################
## Your Code Here
############################